# Compare wake-word models: personal and general

Scores every `server/models/jarvie_<MODE>*.onnx` on held-out real recordings, per speaker. Set `MODE` below to
`wake` (stage 1), `name` (stage 2, "jarvie" alone) or `sleep`:
- **personal**: your test set (`~/jarvie-data/audio/<MODE>/`, recorded with `record_session.py --split test`)
- **general**: other people's test sets (`~/jarvie-data/audio-speakers/<name>/<MODE>/`, `--split test --speaker <name>`),
  averaged so every speaker counts equally. Use people who are in **no** training data.

Same definitions as `wakeword_eval.ipynb`: each model's threshold is the equal-error-rate threshold on normal-pace
clips; fast/slow recall and how often each similarity tier fires are measured at that threshold. Clips are padded with
1 s of silence on both sides, as livekit's `predict_clip` does. The run refuses to score a test clip that also appears in
`~/jarvie-data/real/` (training data).

Model families (from `train_wakeword_runpod.ipynb`): `B` synthetic (M1), `B+real-v1` synthetic + your recordings (M2),
`B+real-v1-nosyn` recordings only (M3), `B+real-v1-syn10__ft` B fine-tuned on recordings (M4).

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# The server/ folder, found from wherever Jupyter started (models/ may not exist yet).
SERVER = next(
    p
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "jarvie").is_dir() and (p / "pyproject.toml").is_file()
)
sys.path.insert(0, str(SERVER))
from jarvie.eval import compare  # noqa: E402

pd.set_option("display.precision", 3)
# Which model family: "wake" (stage 1: "jarvie, wake up"), "name" (stage 2: "jarvie" alone) or "sleep".
MODE = "wake"
MODELS = sorted((SERVER / "models").glob(f"jarvie_{MODE}*.onnx"))
TEST_SETS = compare.discover_test_sets(MODE)  # {"me": ..., "<other speaker>": ...}
# Gate table: the false-fire budget, in false fires per hour of everyday speech.
MAX_FPPH = 5.0

# Clips that say "jarvie, I need some help": the old model was trained on it, A/B weren't (they use
# "jarvie, help me"), so including them compares the models on a phrase only one of them knows.
EXCLUDE = (
    {f"pos_desk_loud-normal_wake_{i:03d}.wav" for i in range(4, 8)} if MODE == "wake" else set()
)
# Negatives that are really acceptable wake phrases, scored as positives: put the "jarvie wakie wake"
# clips here, and the "jarvis wake up" ones too if Jarvis should wake Jarvie.
AS_POSITIVE: set[str] = set()

print("models:", *[m.name for m in MODELS], sep="\n  ")
print("test sets:", *[f"{k}: {v}" for k, v in TEST_SETS.items()], sep="\n  ")
missing = compare.missing_inputs(MODE, MODELS)
print("ready to evaluate" if not missing else "NOT READY:\n  " + "\n  ".join(missing))

In [ ]:
# Scores every model on every test set and builds all the tables below. Refuses to run until the
# models and your test recordings exist, or if a test clip is also in ~/jarvie-data/real/ (training).
result = compare.evaluate(MODE, MODELS, exclude=EXCLUDE, as_positive=AS_POSITIVE, max_fpph=MAX_FPPH)
scores = result.scores
result.counts  # clips per speaker behind each number

## Personal: your voice
Lower `eer_normal` and `fires_*` are better; higher recall is better. This table picks the **personal model**.

In [ ]:
personal = result.personal
personal.T

## Gate: false fires per hour
For the stage-2 gate in front of Deepgram (`server/training/STAGE2_GATE.md`). Uses your everyday-speech test
recordings (`neg-everyday`, in `~/jarvie-data/audio/wake/` whatever `MODE` is), counting separate fires.
- `fpph_at_threshold`: false fires per hour at the Personal table's threshold
- `gate_threshold` / `gate_recall` / `gate_fpph`: the best recall (all paces) within `MAX_FPPH`

A fire is counted as one gate opening (it holds the gate open ~3 s for the transcript check), so a model
stuck above the threshold counts once per 3 s. These long recordings are left out of every other table.
Check `everyday_hours`: under an hour, one fire more or less moves the rate a lot. NaN means no everyday
audio, or no threshold stays within the budget.

In [ ]:
result.gate.T

## General: other people's voices
Averaged over speakers. This table picks the **deployment model**; it should beat the synthetic-only model here.
Empty until you have test recordings from at least one other person.

In [ ]:
general = result.general
general.T if not general.empty else "no other speakers' test recordings yet"

In [ ]:
# Every speaker separately: spot a model that works for some voices and not others.
result.by_speaker[
    [
        "threshold",
        "eer_normal",
        "recall_normal",
        "recall_fast",
        "recall_slow",
        "fires_similar",
        "fires_very-similar",
    ]
]

## Per-clip scores\nWhere models disagree, listen to the clip: it usually explains the difference.

In [ ]:
per_clip = scores.pivot_table(
    index=["speaker", "kind", "similarity", "pace", "file"], columns="model", values="max_score"
)
per_clip.style.background_gradient(axis=None, cmap="RdYlGn")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.5), squeeze=False)
cols = ["recall_normal", "recall_fast", "recall_slow", "fires_similar", "fires_very-similar"]
for ax, (title, table) in zip(
    axes[0], [("Personal (your voice)", personal), ("General (other voices)", general)]
):
    if table.empty:
        ax.set_axis_off()
        ax.set_title(f"{title}: no data yet")
        continue
    table[cols].T.plot.bar(ax=ax, rot=0, legend=False)
    ax.set(title=title, ylim=(0, 1))
axes[0][0].legend(fontsize=7)
plt.tight_layout()
plt.show()